# Pareto Optimization of SEI Additives with ALCHEMI

Lithium-metal and lithium-ion batteries depend on interfaces. During cycling, electrolyte molecules touch highly reactive electrode surfaces: the negative electrode can drive reduction chemistry, while the positive electrode can drive oxidation chemistry. Some decomposition is harmful because it consumes electrolyte and active lithium, but some early decomposition can be useful when it forms a thin passivating interphase ([Shi et al.](https://www.nature.com/articles/s41524-018-0064-0)).

At the anode, this protective layer is usually called the solid electrolyte interphase (SEI). A useful SEI should block electrons, allow Li+ transport, remain chemically and mechanically stable, and avoid continuously attacking the electrolyte ([Shi et al.](https://www.nature.com/articles/s41524-018-0064-0)). Electrolyte additives are often chosen because they react before the bulk solvent and help seed a more protective SEI ([Balakrishnan et al.](https://www.sciencedirect.com/science/article/pii/S2451910320300089)). The design tension is subtle: an additive that barely interacts may do nothing, while one that binds or decomposes too aggressively may create impedance, gas, or unstable products.

A realistic battery-interface simulation would need explicit liquid electrolyte, electron transfer, lithium-ion motion, voltage, many reaction pathways, long timescales, and multiple surface structures. The breadth of these modeling challenges is a central theme in the SEI modeling literature ([Shi et al.](https://www.nature.com/articles/s41524-018-0064-0)). Instead, this notebook asks you to build a small, transparent screening proxy using the skills from Part 1: generate surface+molecule systems, relax them in ALCHEMI Toolkit batches, compute binding energies, and use those energies to rank candidates.

The two challenge objectives encode the tradeoff. First, a molecule should have useful moderate interaction with a reactive Li-metal proxy, which represents SEI seeding. Second, once a passivating SEI-like product exists, the molecule should interact weakly with that surface, which represents compatibility with a protective layer. Because these goals can conflict, you will use Pareto hypervolume improvement rather than a single hand-picked threshold.

The chemistry here is intentionally simplified. Li metal is a reactive anode proxy. Each molecule class maps to one passivating SEI-product proxy surface using the lookup table in `data/class_surface_lookup.csv`. The bundled structures are a starter panel for a workflow exercise, not production battery-interface reference models. You should add electrolyte or additive molecules from the literature by providing your own structure and manifest row, and record the citation/provenance for each custom molecule. For example, FEC is included because it is a widely studied SEI additive with reported LiF-containing reduction products ([Chen et al.](https://www.pnnl.gov/publications/reduction-mechanism-fluoroethylene-carbonate-stable-solid-electrolyte-interphase-film)).

The reward functions are deliberately simple but no longer tied to one arbitrary target energy. They use a moderate-adsorption window for Li-metal seeding and a weak-adsorption preference for SEI passivation, following the same qualitative logic used in SEI-additive and Sabatier-style surface-screening literature ([Lee et al.](https://www.frontiersin.org/journals/energy-research/articles/10.3389/fenrg.2021.654460/full)). The exact constants remain a challenge calibration so the task is reproducible and model-free to grade.


## What You Need To Produce

Write `outputs/challenge_submission.csv` with one row per molecule you evaluate and these columns:

`candidate_id`, `role`, `molecule_class`, `passivating_surface_id`, `E_bind_Li_eV`, `E_bind_passivating_eV`, `seeding_score`, `passivation_score`, `is_pareto`, `hypervolume_improvement`, `selected`.

Optional but recommended: also write `outputs/raw_component_energies.csv` so the grader can check your binding-energy arithmetic without running any model calls. Additional candidate/provenance columns are allowed; the grader ignores columns outside the required set.


## Control Panel

These defaults use a conservative FIRE2 relaxation setup for the SEI challenge geometries. You may reduce `TOOLKIT_N_STEPS` while debugging, then rerun with these stricter settings before scoring or submitting.


In [ ]:
from pathlib import Path

TOOLKIT_CHECKPOINT = "medium-mpa-0"
TOOLKIT_HEAD = None
TOOLKIT_DEVICE = "auto"
TOOLKIT_DTYPE = "float32"
TOOLKIT_COMPILE_MODEL = False
TOOLKIT_ENABLE_CUEQ = True
TOOLKIT_DT = 0.005
TOOLKIT_N_STEPS = 5000
TOOLKIT_FMAX = 0.05
TOOLKIT_FIRE2_MAXSTEP = 0.04
TOOLKIT_D3BJ = None
BATCH_SIZE = 2

ADSORPTION_HEIGHT_A = 2.6
LI_METAL_ADSORPTION_HEIGHT_A = 2.1
MIN_ADSORPTION_CLEARANCE_A = 1.6
ADSORPTION_CELL_A = (15.0, 10.0, 30.0)
PASSIVATING_TILE_SPACING_A = (4.0, 3.0)
BOTTOM_LAYER_TOLERANCE_A = 0.35
GAS_BOX_A = 20.0
ADSORPTION_SITE_LIMIT = 3
ADSORPTION_AZIMUTH_ANGLES_DEG = (0.0, 180.0)
FROZEN_SURFACE_FRACTION = 1.0

OUTPUT_DIR = Path("outputs")
SUBMISSION_PATH = OUTPUT_DIR / "challenge_submission.csv"
RAW_COMPONENT_ENERGIES_PATH = OUTPUT_DIR / "raw_component_energies.csv"


## Setup

The [ALCHEMI Toolkit documentation](https://nvidia.github.io/nvalchemi-toolkit/) describes the same core workflow used in Part 1: structures are represented as `AtomicData`, packed into `Batch` objects, evaluated by model wrappers such as MACE, and relaxed with Toolkit dynamics/optimizer components. MACE is a higher-order equivariant message-passing MLIP architecture, and this notebook uses ASE `Atoms` objects for local structure handling. This notebook reuses the Part 1 helper backend so your challenge code focuses on the scientific workflow and bookkeeping.


In [ ]:
import os
import sys
from importlib.metadata import version, PackageNotFoundError

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "data" / "molecule_manifest.csv").exists():
    candidate = NOTEBOOK_DIR / "challenge-sei"
    if (candidate / "data" / "molecule_manifest.csv").exists():
        NOTEBOOK_DIR = candidate.resolve()
    else:
        raise RuntimeError("Start Jupyter from challenge-sei or from the repository root.")
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

REPO_ROOT = NOTEBOOK_DIR.parent
PART1_ROOT = REPO_ROOT / "part-1-batched-adsorption"
if not (PART1_ROOT / "helpers" / "__init__.py").exists():
    raise RuntimeError("Cannot find Part 1 helpers. Keep challenge-sei beside part-1-batched-adsorption.")
sys.path.insert(0, str(PART1_ROOT))

import numpy as np
import pandas as pd
from ase import Atoms
from ase.data import covalent_radii
from ase.geometry import find_mic
from ase.io import read as ase_read
from ase.io import write as ase_write

from helpers import (
    ToolkitRelaxationConfig,
    ToolkitD3BJConfig,
    check_toolkit_native_api,
    get_toolkit_relaxation_engine,
    ase_to_atomic_data,
    atomic_data_to_ase,
    make_active_mask,
    display_widgets_grid,
)
from helpers.config_search import Configuration
from challenge_utils.pareto import dominates, hypervolume_2d, pareto_flags
from challenge_utils.rewards import passivation_score, seeding_score

print(f"Challenge folder : {NOTEBOOK_DIR.name}")
print(f"Part 1 helpers   : {PART1_ROOT.relative_to(REPO_ROOT)}")
for pkg in ("ase", "numpy", "pandas", "torch", "nvalchemi-toolkit", "ovito"):
    try:
        print(f"{pkg:<18}: {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:<18}: not installed")


## 1. Load The Challenge Manifests

Fill in this cell so every molecule has its class-specific passivating surface. Keep the baseline rows (`EC`, `EMC`) because they define the starting Pareto front. The provided manifest is only a starter panel. To add a literature molecule, copy `data/custom_molecule_manifest_template.csv` to `data/custom_molecule_manifest.csv`, add your molecule row, place the corresponding structure under `data/molecules/`, and record the source/provenance in the manifest. Good starting points for choosing additional literature molecules include broad additive reviews such as [Xu et al.](https://www.sciencedirect.com/science/article/pii/S0378775306017538) and [Balakrishnan et al.](https://www.sciencedirect.com/science/article/pii/S2451910320300089), Li-metal SEI reviews such as [Li et al.](https://pmc.ncbi.nlm.nih.gov/articles/PMC5063117/), focused additive papers on [VC-derived SEI](https://www.sciencedirect.com/science/article/abs/pii/S1572665722001187) and [FEC reduction products](https://www.pnnl.gov/publications/reduction-mechanism-fluoroethylene-carbonate-stable-solid-electrolyte-interphase-film).


In [ ]:
# TODO: Load data/molecule_manifest.csv, data/surface_manifest.csv, and
# data/class_surface_lookup.csv with pandas. If data/custom_molecule_manifest.csv
# exists, append those literature/custom rows to the starter molecule table.
# Merge the combined molecule table with the lookup table on molecule_class so
# each molecule has passivating_surface_id.
# Store the merged table in challenge_df.

# molecules_df = ...
# surfaces_df = ...
# lookup_df = ...
# challenge_df = ...

raise NotImplementedError("Load and merge the challenge manifests.")

# Suggested checks after you implement the merge:
# assert challenge_df["passivating_surface_id"].notna().all()
# assert set(challenge_df["role"]) == {"baseline", "additive"}
# display(challenge_df[["candidate_id", "role", "molecule_class", "passivating_surface_id"]])


## 2. Build The Toolkit Relaxation Engine

This is the same native Toolkit path used in Part 1. ALCHEMI exposes batched atomistic data, MLIP model wrappers, hooks, and dynamics components for workflows like geometry optimization. D3 is disabled by default for a compact challenge run; if you enable it for an instructor rerun, record the parameters in your notes.


In [ ]:
status = check_toolkit_native_api()
print(status["message"])
if not status["available"]:
    raise RuntimeError("ALCHEMI Toolkit native API is not available in this kernel.")

if isinstance(TOOLKIT_D3BJ, dict):
    TOOLKIT_D3BJ = ToolkitD3BJConfig(**TOOLKIT_D3BJ)

relaxation_config = ToolkitRelaxationConfig(
    name="toolkit",
    cache_dir=(OUTPUT_DIR / "cache_json").as_posix(),
    use_cached_responses=False,
    toolkit_checkpoint=TOOLKIT_CHECKPOINT,
    toolkit_head=TOOLKIT_HEAD,
    toolkit_device=TOOLKIT_DEVICE,
    toolkit_dtype=TOOLKIT_DTYPE,
    toolkit_compile_model=TOOLKIT_COMPILE_MODEL,
    toolkit_enable_cueq=TOOLKIT_ENABLE_CUEQ,
    toolkit_dt=TOOLKIT_DT,
    toolkit_n_steps=TOOLKIT_N_STEPS,
    toolkit_fmax=TOOLKIT_FMAX,
    toolkit_fire2_maxstep=TOOLKIT_FIRE2_MAXSTEP,
    toolkit_d3bj=TOOLKIT_D3BJ,
    toolkit_require_d3bj=TOOLKIT_D3BJ is not None,
)
RELAXATION_ENGINE = get_toolkit_relaxation_engine(relaxation_config)
print(f"Toolkit relaxation engine ready: {RELAXATION_ENGINE.name}")


## 3. Structure Helpers

These helpers load the bundled structures and place each molecule above the center of a teaching slab using ASE-style structure manipulation ([Larsen et al., 2017](https://doi.org/10.1088/1361-648X/aa680e)). The placement is intentionally minimal: the challenge is about reproducing the Part 1 workflow logic, not building a production adsorption-site search.


In [ ]:
def load_atoms(relative_path):
    """Read an ASE Atoms object from a path relative to the challenge folder."""
    atoms = ase_read(Path(relative_path))
    return atoms


def gas_box(atoms, *, box_A=GAS_BOX_A):
    """Return a centered molecule in a periodic cubic vacuum box."""
    gas = atoms.copy()
    gas.set_cell([box_A, box_A, box_A])
    gas.set_pbc([True, True, True])
    gas.center()
    return gas


def surface_center_xy(surface):
    """Return the in-plane center of the first two periodic cell vectors."""
    cell = np.asarray(surface.cell.array, dtype=float)
    center = 0.5 * (cell[0] + cell[1])
    return center[:2]


def surface_adsorption_xy(surface, surface_id=None, *, top_layer_tolerance_A=0.35):
    """Return the in-plane placement point for a molecule on a surface."""
    if surface_id == "Li_metal":
        return surface_center_xy(surface)
    top_z = float(np.max(surface.positions[:, 2]))
    top_layer = surface.positions[surface.positions[:, 2] >= top_z - top_layer_tolerance_A]
    if len(top_layer):
        return np.mean(top_layer[:, :2], axis=0)
    return surface_center_xy(surface)


def lateral_distance_A(surface, xy_a, xy_b):
    """Return minimum-image distance between two in-plane points."""
    lengths = np.asarray(surface.cell.lengths()[:2], dtype=float)
    delta = np.abs(np.asarray(xy_a, dtype=float) - np.asarray(xy_b, dtype=float))
    periodic_delta = np.minimum(delta, np.maximum(lengths - delta, 0.0))
    return float(np.linalg.norm(periodic_delta))


def adsorption_site_candidates(
    surface,
    surface_id=None,
    *,
    max_sites=ADSORPTION_SITE_LIMIT,
    top_layer_tolerance_A=0.6,
    min_site_separation_A=0.75,
):
    """Generate a compact center/top/bridge adsorption-site search set."""
    center = surface_center_xy(surface)
    top_z = float(np.max(surface.positions[:, 2]))
    top_indices = [
        index for index, atom in enumerate(surface)
        if float(atom.position[2]) >= top_z - top_layer_tolerance_A
    ]
    top_indices.sort(key=lambda index: lateral_distance_A(surface, surface.positions[index, :2], center))

    pool = [{"site_label": "center", "xy": np.asarray(center, dtype=float)}]
    for index in top_indices[:12]:
        atom = surface[index]
        pool.append({
            "site_label": f"top_{atom.symbol}{index}",
            "xy": np.asarray(atom.position[:2], dtype=float),
        })

    bridge_pool = []
    nearest_top = top_indices[:8]
    for left, i in enumerate(nearest_top):
        for j in nearest_top[left + 1:]:
            distance = lateral_distance_A(surface, surface.positions[i, :2], surface.positions[j, :2])
            if distance > 4.5:
                continue
            xy = 0.5 * (surface.positions[i, :2] + surface.positions[j, :2])
            bridge_pool.append((lateral_distance_A(surface, xy, center), i, j, xy))
    for _, i, j, xy in sorted(bridge_pool)[:8]:
        pool.append({"site_label": f"bridge_{i}_{j}", "xy": np.asarray(xy, dtype=float)})

    selected = []
    for site in pool:
        if all(lateral_distance_A(surface, site["xy"], prev["xy"]) >= min_site_separation_A for prev in selected):
            selected.append(site)
        if len(selected) >= max_sites:
            break
    for index, site in enumerate(selected):
        site["site_id"] = f"s{index}"
    return selected


ADSORPTION_ANCHOR_ELEMENTS = {"O", "N", "F", "P", "S"}


def adsorption_height_for_surface(surface_id):
    """Return the target anchor-atom height for a surface."""
    if surface_id == "Li_metal":
        return LI_METAL_ADSORPTION_HEIGHT_A
    return ADSORPTION_HEIGHT_A


def use_anchor_down_orientation(surface_id):
    """Orient adsorbates with a likely binding atom facing the slab."""
    return True


def molecule_anchor_indices(molecule):
    """Prefer chemically interactive hetero atoms when placing the adsorbate."""
    indices = [
        index
        for index, atom in enumerate(molecule)
        if atom.symbol in ADSORPTION_ANCHOR_ELEMENTS
    ]
    return indices or list(range(len(molecule)))


def orient_molecule_for_surface(molecule):
    """Rotate the adsorbate so a stable anchor atom points toward the slab."""
    anchor_indices = molecule_anchor_indices(molecule)
    anchor_index = max(
        anchor_indices,
        key=lambda index: float(np.linalg.norm(molecule.positions[index])),
    )
    anchor_vector = np.asarray(molecule.positions[anchor_index], dtype=float)
    if np.linalg.norm(anchor_vector) > 1e-8:
        molecule.rotate(anchor_vector, [0.0, 0.0, -1.0], center=(0.0, 0.0, 0.0), rotate_cell=False)
    return molecule


def adsorption_cell_matrix():
    """Return the common periodic cell used for all adsorption slabs."""
    return np.diag(np.asarray(ADSORPTION_CELL_A, dtype=float))


def finalize_adsorption_slab(atoms, *, surface_id, source_repeat="custom"):
    """Attach the common adsorption cell and provenance metadata."""
    slab = atoms.copy()
    slab.set_cell(adsorption_cell_matrix(), scale_atoms=False)
    slab.set_pbc([True, True, True])
    slab.info.update(atoms.info)
    slab.info["surface_id"] = surface_id
    slab.info["surface_repeat"] = source_repeat
    return slab


def build_large_lif_slab(surface, *, surface_id):
    """Build a dense LiF teaching slab in the common adsorption cell."""
    target_x, target_y, _ = ADSORPTION_CELL_A
    x_positions = np.arange(1.0, target_x, 2.0)
    y_positions = np.arange(1.0, target_y, 2.0)
    z_layers = sorted({round(float(z), 6) for z in surface.positions[:, 2]})
    symbols = []
    positions = []
    for z in z_layers:
        for iy, y in enumerate(y_positions):
            for ix, x in enumerate(x_positions):
                symbols.append("Li" if (ix + iy) % 2 == 0 else "F")
                positions.append((float(x), float(y), float(z)))
    slab = Atoms(symbols=symbols, positions=positions)
    slab.info.update(surface.info)
    return finalize_adsorption_slab(slab, surface_id=surface_id, source_repeat="motif-tiled")


def build_large_motif_slab(surface, *, surface_id):
    """Tile the left-hand stoichiometric motif into the common adsorption cell."""
    target_x, target_y, _ = ADSORPTION_CELL_A
    spacing_x, spacing_y = PASSIVATING_TILE_SPACING_A
    motif_cutoff_x = 0.5 * float(surface.cell.lengths()[0])
    motif_indices = [index for index, atom in enumerate(surface) if atom.position[0] < motif_cutoff_x]
    motif = surface[motif_indices]
    symbols = []
    positions = []
    for dx in np.arange(0.0, target_x, spacing_x):
        for dy in np.arange(0.0, target_y, spacing_y):
            shifted = motif.positions + np.array([dx, dy, 0.0])
            in_cell = (
                (shifted[:, 0] >= 0.0).all()
                and (shifted[:, 0] < target_x).all()
                and (shifted[:, 1] >= 0.0).all()
                and (shifted[:, 1] < target_y).all()
            )
            if not in_cell:
                continue
            symbols.extend(motif.get_chemical_symbols())
            positions.extend(shifted.tolist())
    slab = Atoms(symbols=symbols, positions=positions)
    slab.info.update(surface.info)
    return finalize_adsorption_slab(slab, surface_id=surface_id, source_repeat="motif-tiled")


def prepare_adsorption_surface(surface, *, surface_id):
    """Return a large, common-cell slab for adsorption and clean references."""
    if surface_id == "Li_metal":
        lengths = np.asarray(surface.cell.lengths()[:2], dtype=float)
        target_xy = np.asarray(ADSORPTION_CELL_A[:2], dtype=float)
        repeats = np.maximum(1, np.ceil(target_xy / lengths).astype(int))
        repeat_tuple = (int(repeats[0]), int(repeats[1]), 1)
        return finalize_adsorption_slab(
            surface.repeat(repeat_tuple),
            surface_id=surface_id,
            source_repeat=f"{repeat_tuple[0]}x{repeat_tuple[1]}x1",
        )
    if surface_id == "LiF":
        return build_large_lif_slab(surface, surface_id=surface_id)
    return build_large_motif_slab(surface, surface_id=surface_id)


def place_molecule_on_surface(
    surface,
    molecule,
    *,
    height_A=ADSORPTION_HEIGHT_A,
    orient_anchor_down=True,
    surface_id=None,
    site_xy=None,
    azimuth_deg=0.0,
):
    """Place a molecule above the slab center using its lowest anchor atom."""
    slab = surface.copy()
    mol = molecule.copy()
    mol.translate(-mol.get_center_of_mass())
    if orient_anchor_down:
        mol = orient_molecule_for_surface(mol)
    if azimuth_deg:
        mol.rotate(float(azimuth_deg), [0.0, 0.0, 1.0], center=(0.0, 0.0, 0.0), rotate_cell=False)
    top_z = float(np.max(slab.positions[:, 2]))
    anchor_indices = molecule_anchor_indices(mol)
    anchor_bottom_z = float(np.min(mol.positions[anchor_indices, 2]))
    bottom_z = float(np.min(mol.positions[:, 2]))
    xy = np.asarray(site_xy, dtype=float) if site_xy is not None else surface_adsorption_xy(slab, surface_id=surface_id)
    z_shift = top_z + height_A - anchor_bottom_z
    min_allowed_z = top_z + MIN_ADSORPTION_CLEARANCE_A
    if bottom_z + z_shift < min_allowed_z:
        z_shift += min_allowed_z - (bottom_z + z_shift)
    mol.translate([xy[0], xy[1], z_shift])
    combined = slab + mol
    combined.set_cell(slab.cell)
    combined.set_pbc(slab.pbc)
    return combined


def surface_active_mask(surface, *, bottom_layer_tolerance_A=BOTTOM_LAYER_TOLERANCE_A):
    """Fix the bottom/subsurface layer and relax the exposed surface layer."""
    bottom_z = float(np.min(surface.positions[:, 2]))
    return [float(atom.position[2]) > bottom_z + bottom_layer_tolerance_A for atom in surface]


def frozen_surface_mask(surface):
    """Compatibility wrapper returning the slab active mask."""
    return surface_active_mask(surface)


def combined_active_mask(surface, combined):
    """Relax the top surface layer and adsorbate while fixing subsurface atoms."""
    return surface_active_mask(surface) + [True] * (len(combined) - len(surface))


def build_sei_config_grid(candidate_id, interaction, surface_id, surface, molecule):
    """Build a Part-1-style site/orientation grid for one SEI adsorption pair."""
    height_A = adsorption_height_for_surface(surface_id)
    orientation = "anchor-down" if use_anchor_down_orientation(surface_id) else "as-loaded"
    configs = []
    for site in adsorption_site_candidates(surface, surface_id=surface_id):
        for azimuth_deg in ADSORPTION_AZIMUTH_ANGLES_DEG:
            atoms = place_molecule_on_surface(
                surface,
                molecule,
                height_A=height_A,
                orient_anchor_down=use_anchor_down_orientation(surface_id),
                surface_id=surface_id,
                site_xy=site["xy"],
                azimuth_deg=azimuth_deg,
            )
            site_label = site["site_label"]
            label = (
                f"{candidate_id}_{interaction}_{surface_id}_"
                f"{site_label}_{orientation}_rot{int(azimuth_deg)}_h{height_A:.1f}"
            )
            configs.append(Configuration(
                label=label,
                host=surface_id,
                adsorbate=candidate_id,
                site=site_label,
                orientation=orientation,
                rot_deg=float(azimuth_deg),
                height=float(height_A),
                atoms=atoms,
                active_mask=combined_active_mask(surface, atoms),
            ))
    return configs


def max_force_from_result(result):
    """Return the maximum force norm stored in a Toolkit result."""
    forces = np.asarray(result.forces, dtype=float).reshape(-1, 3)
    if len(forces) == 0:
        return 0.0
    return float(np.linalg.norm(forces, axis=1).max())


def verify_frozen_atoms_unchanged(job, relaxed_atoms, *, tolerance_A=5e-4):
    """Raise if atoms marked frozen by active_mask moved during relaxation."""
    active_mask = job.get("active_mask")
    if active_mask is None:
        return 0.0
    active = np.asarray(active_mask, dtype=bool)
    frozen = ~active
    if not frozen.any():
        return 0.0
    initial = np.asarray(job["atoms"].positions, dtype=float)
    final = np.asarray(relaxed_atoms.positions, dtype=float)
    displacement = np.linalg.norm(final[frozen] - initial[frozen], axis=1)
    max_displacement = float(displacement.max()) if len(displacement) else 0.0
    if max_displacement > tolerance_A:
        raise RuntimeError(
            f"{job['job_id']}: frozen atoms moved by {max_displacement:.4f} A. "
            "Rerun the helper/settings cells so the slab-freezing mask is active, "
            "then regenerate the relaxation outputs."
        )
    return max_displacement


def reference_bond_pairs(atoms, *, scale=1.25):
    """Infer a conservative set of molecular bonds from the input geometry."""
    pairs = []
    h_indices = [index for index, atom in enumerate(atoms) if atom.symbol == "H"]
    heavy_indices = [index for index, atom in enumerate(atoms) if atom.symbol != "H"]
    for h_index in h_indices:
        candidates = []
        for heavy_index in heavy_indices:
            cutoff = scale * (covalent_radii[atoms[h_index].number] + covalent_radii[atoms[heavy_index].number])
            distance = float(np.linalg.norm(atoms.positions[h_index] - atoms.positions[heavy_index]))
            if distance <= cutoff:
                candidates.append((distance, h_index, heavy_index))
        if candidates:
            distance, i, j = min(candidates)
            pairs.append((min(i, j), max(i, j), distance))
    for i, j in zip(heavy_indices, heavy_indices[1:]):
        cutoff = scale * (covalent_radii[atoms[i].number] + covalent_radii[atoms[j].number])
        distance = float(np.linalg.norm(atoms.positions[i] - atoms.positions[j]))
        if distance <= cutoff:
            pairs.append((i, j, distance))
    return sorted(set(pairs))


def verify_molecule_integrity(job, relaxed_atoms, *, stretch_ratio=1.8, stretch_A=0.8, compress_ratio=0.6):
    """Raise if a mobile adsorbate/reference molecule changes connectivity."""
    reference = job.get("reference_molecule")
    if reference is None:
        return []
    n_molecule = len(reference)
    molecule = relaxed_atoms[-n_molecule:]
    flags = []
    for i, j, start_distance in reference_bond_pairs(reference):
        delta, _ = find_mic(
            molecule.positions[i] - molecule.positions[j],
            cell=relaxed_atoms.cell,
            pbc=relaxed_atoms.pbc,
        )
        final_distance = float(np.linalg.norm(delta))
        if final_distance > max(stretch_ratio * start_distance, start_distance + stretch_A):
            flags.append(
                f"{reference[i].symbol}{i}-{reference[j].symbol}{j} "
                f"stretched {start_distance:.2f}->{final_distance:.2f} A"
            )
        elif final_distance < compress_ratio * start_distance:
            flags.append(
                f"{reference[i].symbol}{i}-{reference[j].symbol}{j} "
                f"compressed {start_distance:.2f}->{final_distance:.2f} A"
            )
    if flags and not job.get("allow_molecule_geometry_failure", False):
        raise RuntimeError(
            f"{job['job_id']}: relaxed molecule geometry is not usable: "
            + "; ".join(flags[:4])
        )
    return flags


def relax_structures(jobs, *, batch_size=BATCH_SIZE, label_prefix="sei_challenge"):
    """Relax job dictionaries in Toolkit batches and return energy/result rows."""
    rows = []
    for start in range(0, len(jobs), batch_size):
        chunk = jobs[start:start + batch_size]
        payloads = [
            ase_to_atomic_data(job["atoms"], structure_id=job["job_id"], active_mask=job.get("active_mask"))
            for job in chunk
        ]
        reply = RELAXATION_ENGINE.relax(payloads, label=f"{label_prefix}_{start // batch_size + 1:03d}")
        for job, result in zip(chunk, reply.atoms):
            relaxed_atoms = atomic_data_to_ase(result)
            frozen_max_displacement_A = verify_frozen_atoms_unchanged(job, relaxed_atoms)
            molecule_geometry_flags = verify_molecule_integrity(job, relaxed_atoms)
            rows.append({
                **{key: value for key, value in job.items() if key not in {"atoms", "active_mask", "reference_molecule"}},
                "energy_eV": float(result.energy),
                "converged": bool(result.converged),
                "optimizer_nsteps": int(result.num_optimization_steps),
                "fmax_eV_A": max_force_from_result(result),
                "frozen_max_displacement_A": frozen_max_displacement_A,
                "molecule_geometry_flags": molecule_geometry_flags,
                "relaxed_atoms": relaxed_atoms,
            })
    return rows


def require_all_converged(results, *, label):
    """Raise with a compact table if any Toolkit relaxation missed TOOLKIT_FMAX."""
    bad_rows = [
        {
            "job_id": row.get("job_id", ""),
            "candidate_id": row.get("candidate_id", ""),
            "interaction": row.get("interaction", ""),
            "surface_id": row.get("surface_id", ""),
            "fmax_eV_A": row.get("fmax_eV_A", np.nan),
            "optimizer_nsteps": row.get("optimizer_nsteps", np.nan),
        }
        for row in results
        if not bool(row.get("converged", False))
    ]
    if bad_rows:
        bad_df = pd.DataFrame(bad_rows).sort_values("fmax_eV_A", ascending=False)
        display(bad_df)
        raise RuntimeError(
            f"{label} has {len(bad_rows)} unconverged relaxation(s). "
            f"Increase TOOLKIT_N_STEPS or reduce TOOLKIT_DT/TOOLKIT_FIRE2_MAXSTEP before scoring."
        )


def select_lowest_energy_site_results(results):
    """Return one lowest-energy reliable adsorption start per candidate/surface pair."""
    df = pd.DataFrame(results)
    if df.empty:
        raise RuntimeError("No adsorption site-search results were produced.")
    df["molecule_geometry_ok"] = df["molecule_geometry_flags"].apply(lambda flags: len(flags) == 0)
    df["reliable_for_minimum"] = df["converged"].astype(bool) & df["molecule_geometry_ok"].astype(bool)

    selected_rows = []
    failure_rows = []
    for key, group in df.groupby(["candidate_id", "interaction"], sort=False):
        reliable = group[group["reliable_for_minimum"]]
        if reliable.empty:
            best_attempt = group.sort_values("energy_eV").iloc[0]
            failure_rows.append({
                "candidate_id": key[0],
                "interaction": key[1],
                "n_starts": int(len(group)),
                "n_converged": int(group["converged"].sum()),
                "best_attempt_job_id": best_attempt.get("job_id", ""),
                "best_attempt_flags": "; ".join(best_attempt.get("molecule_geometry_flags", [])),
            })
            continue
        selected_rows.append(reliable.loc[reliable["energy_eV"].idxmin()].to_dict())

    if failure_rows:
        display(pd.DataFrame(failure_rows))
        raise RuntimeError("At least one molecule/surface pair has no converged, intact adsorption start.")

    selected_df = pd.DataFrame(selected_rows).sort_values(["candidate_id", "interaction"]).reset_index(drop=True)
    summary = selected_df[[
        "candidate_id", "interaction", "surface_id", "site_label", "azimuth_deg", "energy_eV", "fmax_eV_A"
    ]].copy()
    print("Selected lowest-energy reliable adsorption starts:")
    display(summary)
    return selected_df.to_dict("records")


## 4. Build And Relax The Jobs

You need three groups of energies, following the component-energy bookkeeping common to adsorption-style screening workflows. The relaxation path uses the batched ALCHEMI Toolkit pattern described in the [Toolkit documentation](https://nvidia.github.io/nvalchemi-toolkit/). As in Part 1, combined adsorption systems should be generated as a small site/orientation grid and reduced to the lowest-energy reliable relaxed start for each molecule/surface pair.

1. Gas molecules: `E_species`.
2. Clean surfaces: `E_surface` for `Li_metal` and every passivating surface used by the candidates.
3. Combined systems: `E_surface+species` for molecule on Li metal and molecule on its class-specific passivating surface.


In [ ]:
# TODO: Build gas_jobs, clean_surface_jobs, and combined_jobs, then relax them
# with relax_structures(...). Keep enough metadata in each job to recover the
# candidate_id, interaction, and surface_id after relaxation.
#
# Hints:
# - load molecules from challenge_df["structure_path"].
# - load surfaces from surfaces_df["structure_path"], then make one
#   prepare_adsorption_surface(surface, surface_id=surface_id) slab for each
#   surface_id used by the challenge.
# - use gas_box(...) for isolated molecule references.
# - use the prepared adsorption slab, not the raw small surface file, for both
#   clean_surface_jobs and combined_jobs so binding-energy references match.
# - for combined systems, use build_sei_config_grid(...) to generate a small
#   Part-1-style site/orientation search grid for each molecule/surface pair.
# - store each config as one combined job, including site_label,
#   start_orientation, azimuth_deg, and allow_molecule_geometry_failure=True.
# - clean surfaces should use surface_active_mask(surface).
# - include reference_molecule for gas and combined jobs so the geometry audit
#   can flag broken adsorbate starts after relaxation.

# gas_results = relax_structures(gas_jobs, label_prefix="sei_gas")
# clean_surface_results = relax_structures(clean_surface_jobs, label_prefix="sei_clean_surface")
# all_combined_results = relax_structures(combined_jobs, label_prefix="sei_combined")
# combined_results = select_lowest_energy_site_results(all_combined_results)
# require_all_converged(gas_results, label="gas references")
# require_all_converged(clean_surface_results, label="clean-surface references")
# require_all_converged(combined_results, label="selected combined adsorption systems")

raise NotImplementedError("Build and relax the gas, clean-surface, and combined jobs.")


## 5. Compute Binding Energies

Use the same adsorption-energy convention as Part 1:

`E_bind = E_surface+species - E_surface - E_species`

Negative values mean exothermic binding in this challenge convention. The isolated gas molecule is a controlled computational reference, not a full solution-phase free energy; solution SEI modeling would need explicit electrolyte, electrode potential, Li+ coordination, and sampling corrections ([Shi et al.](https://www.nature.com/articles/s41524-018-0064-0)). 

In [ ]:
# TODO: Convert gas_results, clean_surface_results, and combined_results into
# lookup dictionaries, then compute one raw-energy row for each candidate on
# Li_metal and one raw-energy row for the candidate's passivating surface.
# Store the result in raw_component_energies_df with columns:
# candidate_id, interaction, surface_id, E_surface_species_eV, E_surface_eV, E_species_eV

# raw_component_energies_df = ...

raise NotImplementedError("Compute the raw component energy table.")

# OUTPUT_DIR.mkdir(exist_ok=True)
# raw_component_energies_df.to_csv(RAW_COMPONENT_ENERGIES_PATH, index=False)
# display(raw_component_energies_df.head())


In [ ]:
# TODO: Use raw_component_energies_df to compute E_bind_Li_eV and
# E_bind_passivating_eV for every row in challenge_df. Store the table in
# binding_df and keep the required metadata columns.

# binding_df = ...

raise NotImplementedError("Compute binding energies from component energies.")

# display(binding_df[["candidate_id", "role", "E_bind_Li_eV", "E_bind_passivating_eV"]])


## 5b. Visual Inspect Relaxed Adsorption Geometries With OVITO

Before turning relaxed energies into scores, inspect the relaxed surface+molecule geometries. This is where OVITO is most useful: it can catch obvious bad placements, detached fragments, molecules crossing periodic boundaries, or suspicious relaxed geometries before those structures enter the Pareto analysis. The cell writes every relaxed combined structure to `outputs/ovito_structures/` as `extxyz`, then tries to show a small OVITO widget grid in the notebook. If the OVITO widget stack is unavailable in your environment, open the saved `extxyz` files directly in OVITO. See the [OVITO Python documentation](https://www.ovito.org/docs/current/python/) and Stukowski's OVITO paper ([doi:10.1088/0965-0393/18/1/015012](https://doi.org/10.1088/0965-0393/18/1/015012)).


In [ ]:
OVITO_STRUCTURE_DIR = OUTPUT_DIR / "ovito_structures"


def safe_structure_name(*parts):
    """Return a filesystem-safe structure stem from metadata fields."""
    text = "_".join(str(part) for part in parts if str(part))
    return "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in text)


def write_ovito_inspection_structures(results, *, output_dir=OVITO_STRUCTURE_DIR):
    """Write relaxed combined structures as EXTXYZ files for OVITO inspection."""
    output_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for result in results:
        if result.get("interaction") not in {"li_metal", "passivating"}:
            continue
        candidate_id = result["candidate_id"]
        interaction = result["interaction"]
        surface_id = result["surface_id"]
        atoms = result["relaxed_atoms"].copy()
        atoms.info["candidate_id"] = candidate_id
        atoms.info["interaction"] = interaction
        atoms.info["surface_id"] = surface_id
        atoms.info["energy_eV"] = float(result["energy_eV"])
        path = output_dir / f"{safe_structure_name(candidate_id, interaction, surface_id)}.extxyz"
        ase_write(path, atoms, format="extxyz")
        rows.append({
            "candidate_id": candidate_id,
            "interaction": interaction,
            "surface_id": surface_id,
            "energy_eV": float(result["energy_eV"]),
            "converged": bool(result["converged"]),
            "structure_path": path.as_posix(),
        })
    if not rows:
        raise RuntimeError("No relaxed combined structures were found in combined_results.")
    return pd.DataFrame(rows).sort_values(["candidate_id", "interaction"]).reset_index(drop=True)


def choose_inspection_candidates(binding_table, *, max_candidates=6):
    """Choose baseline rows plus binding-energy outliers for visual inspection."""
    baseline_ids = binding_table.loc[
        binding_table["role"].eq("baseline"), "candidate_id"
    ].tolist()
    strongest_li_ids = binding_table.sort_values("E_bind_Li_eV").head(2)["candidate_id"].tolist()
    weakest_passivating_ids = binding_table.sort_values(
        "E_bind_passivating_eV", ascending=False
    ).head(2)["candidate_id"].tolist()
    ordered = [*baseline_ids, *strongest_li_ids, *weakest_passivating_ids]
    return list(dict.fromkeys(ordered))[:max_candidates]


inspection_df = write_ovito_inspection_structures(combined_results)
INSPECT_CANDIDATE_IDS = choose_inspection_candidates(binding_df)
print(f"Wrote {len(inspection_df)} OVITO structure file(s) to {OVITO_STRUCTURE_DIR}")
print("Inspecting:", ", ".join(INSPECT_CANDIDATE_IDS))
display(inspection_df[inspection_df["candidate_id"].isin(INSPECT_CANDIDATE_IDS)])

widget_rows = []
for candidate_id in INSPECT_CANDIDATE_IDS:
    row_widgets = []
    for interaction in ("li_metal", "passivating"):
        matches = inspection_df[
            inspection_df["candidate_id"].eq(candidate_id)
            & inspection_df["interaction"].eq(interaction)
        ]
        if matches.empty:
            continue
        row = matches.iloc[0]
        label = f"{candidate_id} | {row.interaction} | {row.surface_id}"
        row_widgets.append((label, row.structure_path))
    if row_widgets:
        widget_rows.append(row_widgets)

try:
    display_widgets_grid(widget_rows, width="390px", height="310px", show_cell=True)
except Exception as exc:
    print(f"OVITO widget display unavailable: {type(exc).__name__}: {exc}")
    print("Open these EXTXYZ files directly in OVITO instead:")
    display(inspection_df[["candidate_id", "interaction", "surface_id", "structure_path"]])


## 6. Compute Literature-Motivated Reward Scores

The reward functions are provided in `challenge_utils.rewards`. 
They convert each binding energy into an adsorption-strength magnitude, `strength = max(0, -E_bind)`, then apply two bounded rewards.

The Li-metal seeding reward follows a Sabatier-style idea from surface chemistry: useful interaction should be neither too weak nor too strong. It gives full reward for moderate adsorption strengths from 0.8 to 1.5 eV, tapers to zero below 0.5 eV, and tapers to zero above 2.0 eV. 

The passivation reward follows the SEI picture that a dense, intact interphase should suppress continued electrolyte reduction. It gives full reward for weak or endothermic adsorption on the SEI proxy, `strength <= 0.3 eV`, and reaches zero by `0.8 eV`.

These constants are a transparent screening calibration, not universal battery chemistry. 
The bounds are chosen by mapping broad literature ideas onto this small adsorption exercise: Balakrishnan et al. motivate the seeding objective because electrolyte additives can form protective electrode films before the main solvent continues reacting; Shi et al. motivate the passivation objective because a useful SEI should electronically block/passivate further electrolyte reduction; Lee et al. motivate the window shape because Sabatier-style surface descriptors reward binding that is neither too weak nor too strong; and adsorption-energy reviews give broad physical-adsorption versus chemical-adsorption energy scales in kJ/mol.

Using the standard conversion `1 eV per molecule = 96.485 kJ/mol`, the challenge bounds correspond approximately to `0.3 eV = 29 kJ/mol`, `0.5 eV = 48 kJ/mol`, `0.8 eV = 77 kJ/mol`, `1.5 eV = 145 kJ/mol`, and `2.0 eV = 193 kJ/mol`. 
`<=0.3 eV` is treated as weak adsorption suitable for an already-passivating SEI proxy; `0.8-1.5 eV` is treated as a moderate chemisorption-like window for Li-metal seeding; `<0.5 eV` is too weak to seed much chemistry; and `>2.0 eV` is penalized as overbinding.

Background links for score functions: [electrolyte additives and protective films](https://www.sciencedirect.com/science/article/pii/S2451910320300089), [SEI passivation and electron blocking](https://www.nature.com/articles/s41524-018-0064-0), [Sabatier-style adsorption tradeoffs](https://www.frontiersin.org/journals/energy-research/articles/10.3389/fenrg.2021.654460/full), and [adsorption energy scales](https://link.springer.com/article/10.1007/s11696-025-04218-x).


In [ ]:
# TODO: Apply the provided scalar reward functions to the binding energies.
# scored_df = binding_df.copy()
# scored_df["seeding_score"] = scored_df["E_bind_Li_eV"].map(seeding_score)
# scored_df["passivation_score"] = scored_df["E_bind_passivating_eV"].map(passivation_score)

raise NotImplementedError("Compute challenge reward scores.")


## 7. Pareto Front And Hypervolume Improvement

Treat both scores as objectives to maximize. The baseline front is built from `EC` and `EMC`. For each additive, compute how much the 2D dominated hypervolume increases when that additive is added to the baseline front. Use reference point `(0, 0)`.


In [ ]:
# Pareto helpers are imported from challenge_utils.pareto so you can focus on
# applying them to your challenge results rather than implementing the geometry.
# TODO: Add is_pareto and hypervolume_improvement columns to scored_df using
# pareto_flags(...) and hypervolume_2d(...).
# final_df = ...

raise NotImplementedError("Compute Pareto flags and hypervolume improvements.")


## 8. Select Your Additive And Submit

Mark exactly one additive as `selected=True`: the additive with the maximum hypervolume improvement. Baseline rows should not be selected.


In [ ]:
# TODO: Create a boolean selected column in final_df. Exactly one additive should
# be True, and it should have the maximum hypervolume_improvement.
# submission = final_df[[
#     "candidate_id", "role", "molecule_class", "passivating_surface_id",
#     "E_bind_Li_eV", "E_bind_passivating_eV", "seeding_score",
#     "passivation_score", "is_pareto", "hypervolume_improvement", "selected",
# ]].copy()

raise NotImplementedError("Select the final additive and build the submission table.")


In [ ]:
required_columns = [
    "candidate_id", "role", "molecule_class", "passivating_surface_id",
    "E_bind_Li_eV", "E_bind_passivating_eV", "seeding_score",
    "passivation_score", "is_pareto", "hypervolume_improvement", "selected",
]
missing = [column for column in required_columns if column not in submission.columns]
if missing:
    raise RuntimeError(f"Submission is missing required columns: {missing}")
if int(submission["selected"].sum()) != 1:
    raise RuntimeError("Exactly one row must be selected.")

OUTPUT_DIR.mkdir(exist_ok=True)
submission[required_columns].to_csv(SUBMISSION_PATH, index=False)
print(f"Wrote {SUBMISSION_PATH}")
display(submission[required_columns])


## References And Further Reading

- NVIDIA [ALCHEMI Toolkit documentation](https://nvidia.github.io/nvalchemi-toolkit/) for `AtomicData`, `Batch`, model wrappers, and Toolkit dynamics.
- Batatia et al., [MACE: Higher Order Equivariant Message Passing Neural Networks for Fast and Accurate Force Fields](https://openreview.net/forum?id=YPpSngE-ZU), NeurIPS 2022.
- Larsen et al., [The Atomic Simulation Environment - a Python library for working with atoms](https://doi.org/10.1088/1361-648X/aa680e), J. Phys.: Condens. Matter 2017.
- Stukowski, [Visualization and analysis of atomistic simulation data with OVITO - the Open Visualization Tool](https://doi.org/10.1088/0965-0393/18/1/015012), Modelling Simul. Mater. Sci. Eng. 2010.
- Leung et al., [Stability of Solid Electrolyte Interphase Components on Lithium Metal and Reactive Anode Material Surfaces](https://doi.org/10.1021/acs.jpcc.5b11719), J. Phys. Chem. C 2016; examples use large periodic SEI/Li cells and matching slab-interface references.
- Chanussot et al., [The Open Catalyst 2020 Dataset and Community Challenges](https://doi.org/10.1021/acscatal.0c04525), ACS Catalysis 2021; summarizes common slab-adsorbate setup, adsorption-energy references, vacuum, and fixed subsurface atoms.
- Shi et al., [Review on modeling of the anode solid electrolyte interphase (SEI) for lithium-ion batteries](https://www.nature.com/articles/s41524-018-0064-0), npj Computational Materials 2018.
- Xu et al., [A review on electrolyte additives for lithium-ion batteries](https://www.sciencedirect.com/science/article/pii/S0378775306017538), J. Power Sources 2007.
- Balakrishnan et al., [Electrolyte additives for improved lithium-ion battery performance and overcharge protection](https://www.sciencedirect.com/science/article/pii/S2451910320300089), 2020.
- Li et al., [A Review of Solid Electrolyte Interphases on Lithium Metal Anode](https://pmc.ncbi.nlm.nih.gov/articles/PMC5063117/), Advanced Science 2016.
- [Insights into the efficient roles of solid electrolyte interphase derived from vinylene carbonate additive in rechargeable batteries](https://www.sciencedirect.com/science/article/abs/pii/S1572665722001187), 2022.
- Zhang et al., [Reduction Mechanism of Fluoroethylene Carbonate for Stable Solid-Electrolyte Interphase Film on Silicon Anode](https://www.pnnl.gov/publications/reduction-mechanism-fluoroethylene-carbonate-stable-solid-electrolyte-interphase-film), ChemSusChem 2013.
- Lee et al., [The Sabatier Principle in Electrocatalysis: Basics, Limitations, and Extensions](https://www.frontiersin.org/journals/energy-research/articles/10.3389/fenrg.2021.654460/full), Frontiers in Energy Research 2021.
- Aich et al., [Determination of thermodynamic parameters in adsorption studies: a review](https://link.springer.com/article/10.1007/s11696-025-04218-x), Chemical Papers 2025.
- [Hypervolume bibliography](https://hypervolume.org/bibliography.html) for Pareto hypervolume indicator references.
